In [1]:
import pandas as pd
from pathlib import Path

In [3]:
BASE_DIR = Path("02_eda_cleaning.py").resolve().parents[1]

RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "clinical_trials_octreotide.xlsx"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "clinical_trials_octreotide_cleaned.xlsx"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
df = pd.read_excel(RAW_DATA_PATH)

print(f"Shape of raw dataset: {df.shape}")


Shape of raw dataset: (100, 9)


In [6]:
df.head()

,nct_id,title,status,start_date,completion_date,phase,conditions,sponsor,enrollment
0,NCT02874326,Octreotide in Patients With GI Bleeding Due to...,UNKNOWN,2016-10,2018-10,PHASE2,"Hereditary Hemorrhagic Telangiectasia, Gastroi...",Radboud University Medical Center,15.0
1,NCT02119884,Hemodynamic Effects of Terlipressin and High D...,COMPLETED,2014-02,2016-06,PHASE4,Gastric and Esophageal Varices,Shanghai Zhongshan Hospital,88.0
2,NCT02335580,Effect of Portal Vein Thrombosis on the Progno...,COMPLETED,2014-12,2022-12,NaN,"Liver Cirrhosis, Portal Vein, Venous Thrombosi...",General Hospital of Shenyang Military Region,475.0
3,NCT04997317,Treatment of Recurrent or Progressive Meningio...,RECRUITING,2021-04-21,2026-12-31,"PHASE1, PHASE2",Meningioma,"University Hospital, Basel, Switzerland",18.0
4,NCT06345079,Cessation of Somatostatin Analogues After PRRT...,RECRUITING,2024-10-14,2028-06,PHASE2,Neuroendocrine Tumors,Australasian Gastro-Intestinal Trials Group,78.0


In [7]:
df.columns

Index(['nct_id', 'title', 'status', 'start_date', 'completion_date', 'phase',
       'conditions', 'sponsor', 'enrollment'],
      dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   nct_id           100 non-null    object 
 1   title            100 non-null    object 
 2   status           100 non-null    object 
 3   start_date       100 non-null    object 
 4   completion_date  93 non-null     object 
 5   phase            79 non-null     object 
 6   conditions       100 non-null    object 
 7   sponsor          100 non-null    object 
 8   enrollment       99 non-null     float64
dtypes: float64(1), object(8)
memory usage: 7.2+ KB


In [9]:
missing_values = df.isna().sum().sort_values(ascending=False)
print(missing_values)

phase              21
completion_date     7
enrollment          1
nct_id              0
title               0
status              0
start_date          0
conditions          0
sponsor             0
dtype: int64


In [10]:
print(df.duplicated().sum())

0


In [11]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("/", "_")
)

print("Cleaned column names:")
print(df.columns.tolist())

Cleaned column names:
['nct_id', 'title', 'status', 'start_date', 'completion_date', 'phase', 'conditions', 'sponsor', 'enrollment']


In [12]:
before_rows = len(df)
df = df.drop_duplicates()
after_rows = len(df)

print(f"\nRemoved {before_rows - after_rows} duplicate rows.")


Removed 0 duplicate rows.


In [14]:
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()  #Converting values to string and removing extra spaces
    df[col] = df[col].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA}) #Replacing fake missing values with real missing values

In [15]:
print("\nConverting date columns...")

possible_date_columns = [
    "start_date",
    "completion_date",
    "primary_completion_date",
    "study_first_submit_date",
    "last_update_submit_date",
    "results_first_submit_date"
]

for col in possible_date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        print(f"Converted to datetime: {col}")


Converting date columns...
Converted to datetime: start_date
Converted to datetime: completion_date


In [16]:
print("\nCreating trial duration column...")

if "start_date" in df.columns and "completion_date" in df.columns:
    df["trial_duration_days"] = (df["completion_date"] - df["start_date"]).dt.days
    df["trial_duration_months"] = (df["trial_duration_days"] / 30.44).round(1)

    print("Created: trial_duration_days")
    print("Created: trial_duration_months")
else:
    print("Start date or completion date not found. Trial duration was not created.")


Creating trial duration column...
Created: trial_duration_days
Created: trial_duration_months


In [17]:
print("\nCleaning enrollment column...")

if "enrollment_count" in df.columns:
    df["enrollment_count"] = pd.to_numeric(df["enrollment_count"], errors="coerce")
    print("Converted enrollment_count to numeric.")
elif "enrollment" in df.columns:
    df["enrollment"] = pd.to_numeric(df["enrollment"], errors="coerce")
    print("Converted enrollment to numeric.")
else:
    print("No enrollment column found.")


Cleaning enrollment column...
Converted enrollment to numeric.


In [18]:
print("\nCreating study year column...")

if "start_date" in df.columns:
    df["start_year"] = df["start_date"].dt.year
    print("Created: start_year")


Creating study year column...
Created: start_year


In [19]:
df.head()

,nct_id,title,status,start_date,completion_date,phase,conditions,sponsor,enrollment,trial_duration_days,trial_duration_months,start_year
0,NCT02874326,Octreotide in Patients With GI Bleeding Due to...,UNKNOWN,2016-10-01,2018-10-01,PHASE2,"Hereditary Hemorrhagic Telangiectasia, Gastroi...",Radboud University Medical Center,15.0,730.0,24.0,2016.0
1,NCT02119884,Hemodynamic Effects of Terlipressin and High D...,COMPLETED,2014-02-01,2016-06-01,PHASE4,Gastric and Esophageal Varices,Shanghai Zhongshan Hospital,88.0,851.0,28.0,2014.0
2,NCT02335580,Effect of Portal Vein Thrombosis on the Progno...,COMPLETED,2014-12-01,2022-12-01,<NA>,"Liver Cirrhosis, Portal Vein, Venous Thrombosi...",General Hospital of Shenyang Military Region,475.0,2922.0,96.0,2014.0
3,NCT04997317,Treatment of Recurrent or Progressive Meningio...,RECRUITING,NaT,NaT,"PHASE1, PHASE2",Meningioma,"University Hospital, Basel, Switzerland",18.0,NaN,NaN,NaN
4,NCT06345079,Cessation of Somatostatin Analogues After PRRT...,RECRUITING,NaT,2028-06-01,PHASE2,Neuroendocrine Tumors,Australasian Gastro-Intestinal Trials Group,78.0,NaN,NaN,NaN


In [20]:
print("\nChecking study type distribution...")

if "study_type" in df.columns:
    print(df["study_type"].value_counts(dropna=False))
else:
    print("study_type column not found.")


Checking study type distribution...
study_type column not found.


In [23]:
print("\n==============================")
print("KEY CATEGORICAL SUMMARIES")
print("==============================")

categorical_columns_to_check = [
    "overall_status",
    "phase",
    "study_type",
    "intervention_type",
    "lead_sponsor_name",
    "sex",
    "healthy_volunteers"
]

for col in categorical_columns_to_check:
    if col in df.columns:
        print(f"\nTop values in {col}:")
        print(df[col].value_counts(dropna=False).head(10))



KEY CATEGORICAL SUMMARIES

Top values in phase:
phase
PHASE2            33
<NA>              21
PHASE3            19
PHASE4            11
PHASE1             7
PHASE1, PHASE2     5
EARLY_PHASE1       2
PHASE2, PHASE3     2
Name: count, dtype: int64


In [24]:
print("\n==============================")
print("NUMERIC SUMMARIES")
print("==============================")

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

if len(numeric_cols) > 0:
    print(df[numeric_cols].describe())
else:
    print("No numeric columns found.")


NUMERIC SUMMARIES
        enrollment  trial_duration_days  trial_duration_months   start_year
count    99.000000            51.000000              51.000000    63.000000
mean    135.818182          1448.274510              47.582353  2008.714286
std     290.434487          1148.442843              37.727585     5.150724
min       0.000000             0.000000               0.000000  1995.000000
25%      22.000000           715.000000              23.500000  2005.000000
50%      45.000000          1218.000000              40.000000  2008.000000
75%     123.500000          1932.000000              63.450000  2013.000000
max    2000.000000          7032.000000             231.000000  2018.000000


In [25]:
print("\nSaving cleaned dataset...")

df.to_excel(OUTPUT_PATH, index=False)

print(f"Cleaned dataset saved to:")
print(OUTPUT_PATH)

print("\nStage 2 EDA and cleaning completed successfully.")


Saving cleaned dataset...
Cleaned dataset saved to:
E:\Data Science\My R&D\clinical_trials_project\data\processed\clinical_trials_octreotide_cleaned.xlsx

Stage 2 EDA and cleaning completed successfully.
